In [17]:
#### Core script to defines and execute rf runs using functions defined in the following scripts

# config          # configures the model run with input data and defaults (where not directly input)
# spatial_cv      # set up the spatial CV blocks based on fold CSV
# train           # trains model and predicts on test data
# evaluate        # calculates metrics for each fold and overall 
# importance      # calculates feature importance + permutation importance for each fold and overall 

from pathlib import Path
import json
import dataclasses
import pandas as pd

from step_a_config import RunConfig
from step_b_spatial_cv import make_spatial_folds
from step_c_train import train_model
from step_d_evaluate import evaluate
from step_e_importance import get_feature_importance

In [18]:
##### Set-up (doesn't change)

# Get the current working directory
cd = Path.cwd().parent.parent 

# set directories
DATA_DIR = Path(f"{cd}/Data/Clean/Training_data/")
RESULTS_DIR = Path(f"{cd}/Results/RF_models/")
FOLD_DIR = Path(f"{cd}/Data/Fold_assignments/") 

In [19]:
#### DEFINE RUN (MANUAL)

version = 'capital'
name_of_run =  'capital_rf_v1' 

target_var = 'log_capital_intensity_USD_per_tonne' 
non_target_var = 'log_capital_intensity_USD_per_USD'
# log_capital_intensity_USD_per_USD OR log_capital_intensity_USD_per_tonne
# log_labor_intensity_jobs_per_million_USD OR log_labor_intensity_jobs_per_tonne

type_of_model = 'rf' # rf or qrf

In [20]:
##### Configuration (doesn't not change)

# config
fold = FOLD_DIR / f"{version}_folds.csv"
model_data = f"{version}_final.csv"

RUNS = [
    RunConfig(
        run_name         = name_of_run,
        target           = target_var,   
        non_target       = non_target_var,
        dataset          = model_data,
        fold_assignments = fold,
        model_type       = type_of_model,
        id_cols          = ["PROJ_ID", "country_ID"],
    ),
]

In [21]:
##### Define function to save results into defined directory
def save_results(results, config, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True) # creates folder if it doesn't already exist 

    # save config details for run 
    # convert any Path objects to strings so json can handle them
    config_dict = dataclasses.asdict(config)
    config_dict = {
        k: str(v) if isinstance(v, Path) else v
        for k, v in config_dict.items()
    }
    (out_dir / "config.json").write_text(
        json.dumps(config_dict, indent=2)
    )

    # save predictions across all folds
    results["predictions"].to_parquet(out_dir / "predictions.parquet", index=False)

    # save the best parameters for each fold
    params_df = pd.DataFrame([
        {"fold": r["fold"], **r["best_params"]}
        for r in results["fold_results"]
    ])
    params_df.to_csv(out_dir / "best_params.csv", index=False)

    # save metrics and importance scores
    results["metrics"].to_csv(out_dir / "metrics.csv", index=False)
    results["importance"].to_csv(out_dir / "importance.csv", index=False)

    print(f"  saved to {out_dir}")

In [22]:
##### RUN   

# define the function to run all scripts in order 
def run(config):
    print(f"\n{'═'*60}")
    print(f"  run: {config.run_name}")
    print(f"  target: {config.target}  |  model: {config.model_type}")
    print(f"{'═'*60}")

    # load model data
    df = pd.read_csv(DATA_DIR / config.dataset)

    # define which columns are feature columns (exclude ID columns and un-used target columns)
    feature_cols = [
        c for c in df.columns
        if c not in config.id_cols + [config.target] + [config.non_target] 
    ]
    print(f"  features: {len(feature_cols)} columns")

    # run spatial folds functions 
    print("\n── Spatial folds ────────────────────────────────────────")
    folds = make_spatial_folds(df, config)

    # run training and prediction functions
    print("\n── Training ─────────────────────────────────────────────")
    results = train_model(df, feature_cols, folds, config)

    # run evaluation functions
    results["metrics"] = evaluate(results, config)

    # run importance functions
    print("\n── Feature importance ───────────────────────────────────")
    results["importance"] = get_feature_importance(results, df, config)

    # run save function
    print("\n── Saving ───────────────────────────────────────────────")
    out_dir = RESULTS_DIR / config.run_name
    save_results(results, config, out_dir)

# Run run function for each model configuration defined above
for config in RUNS:
    run(config)


════════════════════════════════════════════════════════════
  run: capital_rf_v1
  target: log_capital_intensity_USD_per_tonne  |  model: rf
════════════════════════════════════════════════════════════
  features: 36 columns

── Spatial folds ────────────────────────────────────────
  fold_1: 25 train countries (4,929 rows) | 1 test countries (5,182 rows)
  fold_2: 25 train countries (8,827 rows) | 1 test countries (1,284 rows)
  fold_3: 25 train countries (7,403 rows) | 1 test countries (2,708 rows)
  fold_4: 23 train countries (937 rows) | 3 test countries (9,174 rows)

── Training ─────────────────────────────────────────────

── fold_1 ──────────────────────────────
  test countries: ['BRA']
  best params: {'n_estimators': 200, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': None}

── fold_2 ──────────────────────────────
  test countries: ['CAN']


/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  best params: {'n_estimators': 200, 'min_samples_leaf': 2, 'max_features': 0.3, 'max_depth': 30}

── fold_3 ──────────────────────────────
  test countries: ['USA']
  best params: {'n_estimators': 500, 'min_samples_leaf': 2, 'max_features': 0.5, 'max_depth': 20}

── fold_4 ──────────────────────────────
  test countries: ['BRA', 'CAN', 'USA']
  best params: {'n_estimators': 200, 'min_samples_leaf': 2, 'max_features': 0.3, 'max_depth': 30}

── Evaluation ───────────────────────────────
   fold     rmse      mae        r2   rmse_orig    mae_orig  med_abs_err
 fold_1 1.832867 1.443432  0.090329 1609.426658  445.952442     1.184183
 fold_2 0.779928 0.588078  0.259478  674.575077  299.482453     0.470585
 fold_3 1.356352 1.187447 -1.163328 3510.283707 1388.918529     1.103888
 fold_4 1.472215 1.086947  0.336917 2177.309295  617.048744     0.782309
overall 1.360341 1.076476 -0.119151 1992.898684  687.850542     0.885241

── Feature importance ───────────────────────────────────


/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



── Feature Importance (top 15) ──────────────────────────────
                                  feature  impurity_mean  impurity_std  permutation_mean  permutation_std
           ruminants_share_production_USD       0.042065      0.021482          0.049421         0.021718
         sugar_crops_share_production_USD       0.128700      0.125809          0.046185         0.022352
log_country_capital_intensity_USD_per_USD       0.071936      0.039793          0.043833         0.087666
                        share_large_field       0.033766      0.013436          0.019135         0.009380
             poultry_share_production_USD       0.031715      0.009404          0.015240         0.016863
                           crop_intensity       0.034427      0.011466          0.013423         0.010834
                                      SOC       0.024706      0.002681          0.010248         0.009132
        monogastrics_share_production_USD       0.044288      0.024406          0.009700 